# MD-GRTN Phase 1 — Optimised for MAE ≈ 13

**Key changes from v2:**

| What | Before | Now | Why |
|---|---|---|---|
| `d_model` | 64 | 128 | More capacity for 5 transformer layers |
| `n_heads` | 4 | 4 | 128÷4=32 ✓ |
| Training loss | masked MAE | **Huber** | Paper's loss, more stable |
| LR schedule | ReduceLROnPlateau p=5 | **CosineAnnealing** | Doesn't kill LR too early |
| Normalisation | global axis=(0,1) | **per-sensor axis=0** | Preserves sensor variance |
| BackNet | 2-layer MLP | **3-layer MLP + residual** | Better denoising |
| Output head | Linear(d×S→P) | **Linear→GELU→Linear** | Non-linear projection |
| train_ratio | 0.6 | **0.7** | More training data (paper: 7:1:2) |
| optimizer | Adam | **AdamW wd=0.01** | Better regularisation |
| epochs | 100 | **150** | More time to converge |

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} ✓')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    # data
    data_path   = "/content/PEMS08.npz"
    num_nodes   = 170
    in_features = 3
    seq_len     = 12
    pred_len    = 12
    feature_idx = 0

    # model — bigger capacity
    d_model    = 128        # ↑ from 64  (128÷4=32 per head ✓)
    n_heads    = 4
    num_layers = 5
    gru_layers = 3
    dropout    = 0.1

    # training — paper-aligned
    batch_size  = 32
    lr          = 1e-3
    weight_decay= 0.01      # AdamW regularisation
    epochs      = 150       # ↑ from 100
    patience    = 20        # ↑ from 15
    delta_h     = 1.0       # Huber loss δ
    train_ratio = 0.7       # ↑ from 0.6  (paper: 7:1:2)
    val_ratio   = 0.1

    # checkpoint
    ckpt_path   = 'mdgrtn_v4_ckpt.pt'
    best_path   = 'mdgrtn_v4_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert cfg.d_model % cfg.n_heads == 0, f"{cfg.d_model} not divisible by {cfg.n_heads}"
print(f'Config ready | d_model={cfg.d_model} | n_heads={cfg.n_heads} | head_dim={cfg.d_model//cfg.n_heads}')
print(f'  num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers} | train_ratio={cfg.train_ratio}')
print(f'  device={device}')

In [ ]:
def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    print('Keys:', list(raw.keys()))
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')

    # ── per-sensor z-score (axis=0 over time only) ──
    # Better than global: each sensor has its own scale
    mean = data.mean(axis=0, keepdims=True)   # (1, N, F)
    std  = data.std(axis=0,  keepdims=True) + 1e-8
    data = (data - mean) / std

    for key in ('adjacency_matrix','adj_mx','adj'):
        if key in raw:
            A_dist = raw[key].astype(np.float32)
            print(f'Adjacency loaded from "{key}" shape={A_dist.shape}')
            break
    else:
        print('No adjacency — using identity')
        A_dist = np.eye(N, dtype=np.float32)

    deg    = A_dist.sum(axis=1, keepdims=True) + 1e-8
    A_dist = A_dist / deg
    return data, mean, std, A_dist


class TrafficDataset(Dataset):
    def __init__(self, data, seq_len, pred_len, feature_idx):
        self.data     = torch.from_numpy(data)
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.feat_idx = feature_idx
        self.length   = len(data) - seq_len - pred_len + 1

    def __len__(self): return self.length

    def __getitem__(self, i):
        x = self.data[i : i+self.seq_len]
        y = self.data[i+self.seq_len : i+self.seq_len+self.pred_len, :, self.feat_idx]
        return x, y


def build_dataloaders(cfg):
    set_seed()
    data, mean, std, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))

    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    dl_tr = DataLoader(TrafficDataset(data[:t1],  cfg.seq_len, cfg.pred_len, cfg.feature_idx), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(data[t1:t2],cfg.seq_len, cfg.pred_len, cfg.feature_idx), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(data[t2:],  cfg.seq_len, cfg.pred_len, cfg.feature_idx), shuffle=False, **kw)

    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean, std, A_dist

print('Data utilities ready  (per-sensor z-score normalisation).')

In [ ]:
# BackNet: 3-layer MLP with residual connection
class BackNet(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, in_dim),
        )
        self.norm = nn.LayerNorm(in_dim)

    def forward(self, x):
        return self.norm(x + self.net(x))   # residual connection


class MultiHeadAttentionFusion(nn.Module):
    def __init__(self, in_dim, d_model, n_heads, n_seqs=1):
        super().__init__()
        self.projs  = nn.ModuleList([nn.Linear(in_dim, d_model) for _ in range(n_seqs)])
        self.attn   = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.fc_out = nn.Linear(d_model * n_seqs, d_model)
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, seqs):
        B, S, N, _ = seqs[0].shape
        projected = []
        for proj, seq in zip(self.projs, seqs):
            h = proj(seq.reshape(B*S, N, -1))
            h, _ = self.attn(h, h, h)
            projected.append(h.reshape(B, S, N, -1))
        return self.norm(self.fc_out(torch.cat(projected, dim=-1)))


class MDModule(nn.Module):
    def __init__(self, in_features, d_model, n_seqs=1):
        super().__init__()
        self.backnets = nn.ModuleList([BackNet(in_features, d_model) for _ in range(n_seqs)])
    def forward(self, seqs):
        return [bn(s) for bn, s in zip(self.backnets, seqs)]


class MDAFModule(nn.Module):
    def __init__(self, in_features, d_model, n_heads, n_seqs=1):
        super().__init__()
        self.md  = MDModule(in_features, d_model, n_seqs)
        self.maf = MultiHeadAttentionFusion(in_features, d_model, n_heads, n_seqs)
    def forward(self, seqs):
        return self.maf(self.md(seqs))

print('MDAF defined  (BackNet: 3-layer MLP + residual + LayerNorm).')

In [ ]:
class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)
    def forward(self, x, A):
        return self.lin(torch.einsum('nm,bmd->bnd', A, x))


class GCN_GRU_Layer(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.gcn  = GraphConv(in_dim, hidden_dim)
        self.gru  = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x_seq, A):
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B*N, self.gru.hidden_size, device=x_seq.device)
        outs = []
        for t in range(S):
            gcn_out = F.relu(self.gcn(x_seq[:,t], A))
            h       = self.gru(gcn_out.reshape(B*N,-1), h)
            outs.append(self.norm(h).reshape(B,N,-1))
        return torch.stack(outs, dim=1)


class MGRCModule(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_nodes, num_layers=3):
        super().__init__()
        self.emb      = nn.Parameter(torch.randn(num_nodes, hidden_dim) * 0.01)
        self.adj_conv = nn.Conv2d(2, 1, kernel_size=1)
        dims = [in_dim] + [hidden_dim]*num_layers
        self.layers = nn.ModuleList([
            GCN_GRU_Layer(dims[i], dims[i+1]) for i in range(num_layers)])

    def get_fused_adj(self, A_dist):
        A_dyna  = torch.softmax(self.emb @ self.emb.T, dim=-1)
        stacked = torch.stack([A_dist, A_dyna], dim=0).unsqueeze(0)
        A_F     = F.relu(self.adj_conv(stacked).squeeze(0).squeeze(0))
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)

    def forward(self, x, A_dist):
        A_F = self.get_fused_adj(A_dist)
        for layer in self.layers:
            x = layer(x, A_F)
        return x

print('MGRC defined.')

In [ ]:
class SpatialTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        f    = x.reshape(B*S, N, d)
        h, _ = self.attn(f, f, f)
        h    = self.norm1(f + self.drop(h))
        h    = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, S, N, d)


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        f    = x.permute(0,2,1,3).reshape(B*N, S, d)
        h, _ = self.attn(f, f, f)
        h    = self.norm1(f + self.drop(h))
        h    = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0,2,1,3)

print('Transformer layers defined  (GELU activations).')

In [ ]:
class MDGRTN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        L, h, dr = cfg.num_layers, cfg.n_heads, cfg.dropout
        P, S = cfg.pred_len, cfg.seq_len

        self.mdaf = MDAFModule(F, d, h, n_seqs=1)
        self.mgrc = MGRCModule(d, d, N, num_layers=cfg.gru_layers)

        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, N, d) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, S, 1, d) * 0.02)

        self.spatial_layers  = nn.ModuleList([SpatialTransformerLayer(d, h, dr) for _ in range(L)])
        self.temporal_layers = nn.ModuleList([TemporalTransformerLayer(d, h, dr) for _ in range(L)])

        # Better output head: Linear → GELU → Linear
        self.out_norm = nn.LayerNorm(d)
        self.out_proj = nn.Sequential(
            nn.Linear(d * S, d * 2),
            nn.GELU(),
            nn.Dropout(dr),
            nn.Linear(d * 2, P),
        )

    def forward(self, x_recent, A_dist):
        x = self.mdaf([x_recent])
        x = self.mgrc(x, A_dist)
        x = x + self.spatial_pos
        for layer in self.spatial_layers:
            x = layer(x)
        x = x + self.temporal_pos
        for layer in self.temporal_layers:
            x = layer(x)
        B, S, N, d = x.shape
        x   = self.out_norm(x)
        x   = x.permute(0,2,1,3).reshape(B, N, S*d)
        out = self.out_proj(x)        # (B, N, P)
        return out.permute(0,2,1)     # (B, P, N)


set_seed()
model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready | Parameters: {total:,}')
print(f'  d_model={cfg.d_model} | num_layers={cfg.num_layers} | gru_layers={cfg.gru_layers}')

with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, adj)
print(f'Output shape: {out.shape}  ✓')

In [ ]:
def huber_loss(pred, true, delta=1.0):
    """Paper Eq.27 — training loss."""
    return F.huber_loss(pred, true, delta=delta)

def MAE(pred, true):
    return torch.abs(pred - true).mean()

def RMSE(pred, true):
    return ((pred - true)**2).mean().sqrt()

def MAPE(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

print('Metrics defined  (Huber training loss, MAE/RMSE/MAPE eval).')

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

# per-sensor stats for denorm
mean_flow = torch.from_numpy(mean_np[0,:,cfg.feature_idx]).to(device)  # (N,)
std_flow  = torch.from_numpy(std_np [0,:,cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)

print(f'mean_flow: min={mean_flow.min():.1f}  max={mean_flow.max():.1f}')
print(f'std_flow:  min={std_flow.min():.2f}  max={std_flow.max():.2f}')
print(f'Train={len(dl_train.dataset)} | Val={len(dl_val.dataset)} | Test={len(dl_test.dataset)}')

In [ ]:
def train_epoch(model, loader, optimizer, A_dist, device, cfg):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x, A_dist)
        loss = huber_loss(pred, y, cfg.delta_h)   # Huber loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())
    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = MAE(P, Y).item()
    rmse = RMSE(P, Y).item()
    mape = MAPE(P, Y).item()
    return mae, rmse, mape

print('Train/eval defined  (Huber training, MAE/RMSE/MAPE evaluation).')

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_mae, history, cfg, drive_path=None):
    ckpt = {
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch'          : epoch,
        'best_mae'       : best_mae,
        'history'        : history,
        'seed'           : SEED,
        'd_model'        : cfg.d_model,
        'num_layers'     : cfg.num_layers,
        'gru_layers'     : cfg.gru_layers,
    }
    torch.save(ckpt, cfg.ckpt_path)
    if drive_path:
        import shutil; shutil.copy(cfg.ckpt_path, drive_path)
        print(f'Saved to Drive: {drive_path}  (ep={epoch} mae={best_mae:.3f})')
    else:
        print(f'Checkpoint saved  (ep={epoch} mae={best_mae:.3f})')


def load_checkpoint(model, optimizer, scheduler, cfg, drive_path=None):
    if drive_path:
        import shutil; shutil.copy(drive_path, cfg.ckpt_path)
    if not os.path.exists(cfg.ckpt_path):
        print('No checkpoint — starting fresh.')
        return 1, float('inf'), {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}
    ckpt = torch.load(cfg.ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    print(f'Resumed ep={ckpt["epoch"]} | best_mae={ckpt["best_mae"]:.3f}')
    return ckpt['epoch']+1, ckpt['best_mae'], ckpt['history']

print('Checkpoint utilities ready.')

In [ ]:
# ══════════════════════════════════════════════════
# TRAINING
# ══════════════════════════════════════════════════
set_seed()

optimizer = torch.optim.AdamW(model.parameters(),
                               lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=1e-6)

# ── To resume: uncomment these 3 lines ──
# start_epoch, best_val_mae, history = load_checkpoint(
#     model, optimizer, scheduler, cfg,
#     drive_path='/content/drive/MyDrive/mdgrtn_v4_ckpt.pt')

start_epoch  = 1
best_val_mae = float('inf')
patience_cnt = 0
history      = {'train_loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}

DRIVE_CKPT = None   # ← set Drive path here to auto-save

print(f'Training | epochs={cfg.epochs} | patience={cfg.patience}')
print(f'Optimizer: AdamW lr={cfg.lr} wd={cfg.weight_decay}')
print(f'Schedule:  CosineAnnealing T_max={cfg.epochs} eta_min=1e-6')
print(f'Loss:      Huber δ={cfg.delta_h}')
print(f'Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*68)

for epoch in range(start_epoch, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device, cfg)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  ← best ✓'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch:03d} | lr={lr_now:.2e} | Loss={train_loss:.4f} | '
          f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}%{tag}')

    save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_mae, history, cfg, drive_path=DRIVE_CKPT)

print(f'\nBest Val MAE: {best_val_mae:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue', label='Huber Loss')
axes[0].set_title('Training Loss (Huber)'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', lw=1.5, label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', lw=1.5, label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_v4.png', dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred)
    Y = torch.cat(all_true)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*58)
    print('  TEST RESULTS  —  all 12 steps × 170 sensors')
    print('='*58)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Δ={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Δ={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Δ={mape-8.471:+.3f}%')
    print('='*58)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        pred   = model(x, A_dist)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(MAE(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(RMSE(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(MAPE(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k: np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)